This tutorial showcases a few dataframe manipulations:

We can automatically check whether `df` is single-frame or multi-frame:

In [1]:
from audiospylt import detect_dataframe_type
import pandas as pd

df1 = pd.read_csv('../tsv/conga_multidft.tsv', delimiter='\t')
df2 = pd.read_csv('../tsv/bayle1.tsv', delimiter='\t')

print("df1:", detect_dataframe_type(df1))
print("df2:", detect_dataframe_type(df2))

df1: {'type': 'multi_frame', 'count': 4, 'time_cues': [0.03, 0.1, 0.2, 0.4]}
df2: {'type': 'single_frame', 'pairs': 64, 'freq_cols': ['Frequency (Hz)'], 'amp_cols': ['Amplitude']}


Let’s start with a single frame:

In [2]:
from audiospylt.multiplotter import plot_scatter

plot_scatter(
    files=['../tsv/bayle1.tsv'],
    mode='markers',
    # Frequency-axis scaling:
    freq_axis_mode='linear',  # 'linear' | 'log' | 'mel' | 'mixed'
    freq_axis_mix=0.5,        # used only when freq_axis_mode='mixed'
    mixed_log_floor_hz=1.0,   # used only when freq_axis_mode='mixed'
    # Amplitude-axis scaling:
    amp_axis_mode='linear',   # 'linear' | 'log' | 'mixed'
    amp_axis_mix=0.5,         # used only when amp_axis_mode='mixed'
    amp_log_floor=1e-12,      # used for 'log'/'mixed'
    # Range handling:
    plot_width=1100,
    plot_height=700,
    auto_plot_range=True,
)

It’s a single frame, so it doesn’t include any duration information. We can “freeze” it and extend it (e.g., to 2 seconds):

In [3]:
from audiospylt import expand_to_multi, add_time_cues
import pandas as pd

# Expand single frame
df_single = pd.read_csv('../tsv/bayle1.tsv', delimiter='\t')
df_multi = expand_to_multi(df_single, duration=2.0)

Single-frame and multi-frame dataframes use different TSV formats.

In [4]:
display(df_single)
display(df_multi)

,Frequency (Hz),Amplitude
0,150.000000,0.003061
1,163.157895,0.002791
2,200.000000,0.001160
3,210.526316,0.004657
4,331.578947,0.003659
...,...,...
59,8786.842105,0.000769
60,8800.000000,0.001389
61,8805.263158,0.000994
62,9773.684211,0.000544


,freq_start,freq_stop,time_start,time_stop,amp_min,amp_max
0,150.000000,150.000000,0.0,2.0,0.003061,0.003061
1,163.157895,163.157895,0.0,2.0,0.002791,0.002791
2,200.000000,200.000000,0.0,2.0,0.001160,0.001160
3,210.526316,210.526316,0.0,2.0,0.004657,0.004657
4,331.578947,331.578947,0.0,2.0,0.003659,0.003659
...,...,...,...,...,...,...
59,8786.842105,8786.842105,0.0,2.0,0.000769,0.000769
60,8800.000000,8800.000000,0.0,2.0,0.001389,0.001389
61,8805.263158,8805.263158,0.0,2.0,0.000994,0.000994
62,9773.684211,9773.684211,0.0,2.0,0.000544,0.000544


Let’s visualize the extended frame:

In [5]:
from audiospylt.multiplotter import plot_combined_3d

plot_combined_3d(
    dfs=[df_multi],
    axis_order=("time", "amp", "freq"),
    flip={"amp": True, "time": True},
    plot_width=1100,
    plot_height=700,
)

In the 3D visualization, we can clearly see the static 2-second frame. Next, we’ll add cue points (as a list) and slice the frame at specific positions:

In [6]:
# Add time slices
# This will split the 0-2s segment into 0-0.5, 0.5-1.5, 1.5-2.0
df_sliced = add_time_cues(df_multi, [0.5, 1.5])

In [7]:
plot_combined_3d(
    dfs=[df_sliced],
    axis_order=("time", "amp", "freq"),
    flip={"amp": True, "time": True},
    plot_width=1100,
    plot_height=700,
)

Now that we’ve added cue points, we can apply different effects (like pitch shifting or time stretching) to each segment. We can use the following syntax:


In [8]:
from audiospylt import process_spectral_dataframe


df_processed = process_spectral_dataframe(df_sliced, stretch_factor=[1.0, 2.0, 0.5], pitch_cents=[0, 1500, -250])

plot_combined_3d(
    dfs=[df_processed],
    axis_order=("time", "amp", "freq"),
    flip={"amp": True, "time": True},
    plot_width=1100,
    plot_height=700,
)

If we change `pitch_cents` by large amounts, we can easily hit 0 Hz or exceed the Nyquist frequency. Use these parameters to control what happens:

- Clamp at Nyquist: `process_spectral_dataframe(df, pitch_cents=100, pitch_max_hz=sr/2, pitch_overshoot="clip")`
- Allow overshoot: `process_spectral_dataframe(df, pitch_cents=100, pitch_max_hz=sr/2, pitch_overshoot="allow")`
- Drop out-of-range rows: `process_spectral_dataframe(df, pitch_cents=100, pitch_max_hz=sr/2, pitch_overshoot="threshold")`


In [9]:
# Lets try it on a single frame:

df_single = pd.read_csv('../tsv/bayle1.tsv', delimiter='\t')
df_single_pitched = process_spectral_dataframe(df_single, pitch_factor=10, pitch_min_hz=0, pitch_max_hz=22050, pitch_overshoot="threshold", print_overshoot_freqs=True)

plot_scatter(
    dfs=[df_single_pitched],
    mode='markers',
    # Frequency-axis scaling:
    freq_axis_mode='log',  # 'linear' | 'log' | 'mel' | 'mixed'
    freq_axis_mix=0.5,        # used only when freq_axis_mode='mixed'
    mixed_log_floor_hz=1.0,   # used only when freq_axis_mode='mixed'
    # freq_plot_min_hz=1,     # explicit 0 is only valid for non-log axes
    # Amplitude-axis scaling:
    amp_axis_mode='linear',   # 'linear' | 'log' | 'mixed'
    amp_axis_mix=0.5,         # used only when amp_axis_mode='mixed'
    amp_log_floor=1e-12,      # used for 'log'/'mixed'
    # Range handling:
    plot_width=1100,
    plot_height=700,
    auto_plot_range=True,
)

Overshot frequencies (initial -> pitched):


,freq_initial,freq_pitched
0,2618.421053,26184.210526
1,2631.578947,26315.789474
2,2647.368421,26473.684211
3,2921.052632,29210.526316
4,2934.210526,29342.105263
5,2939.473684,29394.736842
6,2950.000000,29500.000000
7,3894.736842,38947.368421
8,3900.000000,39000.000000
9,3910.526316,39105.263158


Let’s save the processed dataframe:

In [10]:
from audiospylt.io_utils import save_df_tsv

# Save the amp/freq table (peaks_df) for reuse.
save_df_tsv(df_processed, "../tsv/processed.tsv")

Data saved successfully to /home/eggi/Nextcloud/code/public_repos/audiospylt/tsv/processed.tsv at 2026-03-01 17:29:23.741160.


'/home/eggi/Nextcloud/code/public_repos/audiospylt/tsv/processed.tsv'